# W4A16 on Colab T4

1. **Runtime → Change runtime type → T4 GPU** (or GPU, then confirm it is a T4).
2. Run the cells in order.

Colab free often gives the same **NVIDIA T4 (`sm_75`)** this kernel is built for. Sessions disconnect; rebuild if the runtime resets. Use this for CUDA tests and a first bench. Publish README numbers from a dedicated T4 VM if you can.

Get the repo onto this machine with **one** of the next two cells: zip upload (works with uncommitted local work) or `git clone` (only after you push `csrc/` and `setup.py`).

In [ ]:
import torch

assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU, then rerun"
name = torch.cuda.get_device_name(0)
print(name)
print("cuda", torch.version.cuda, "pytorch", torch.__version__)
if "T4" not in name:
    print("Warning: this notebook targets T4 / sm_75; results will not match the project GPU.")

## Option A — upload a zip of the local repo

On your Mac: zip the project (include `csrc/`, `setup.py`, `w4a16/`, `tests/`, `scripts/`). In Colab, upload `repo.zip` with the file browser, then run:

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path("/content/repo.zip")
assert zip_path.is_file(), "Upload repo.zip to /content/repo.zip first"
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall("/content/src")

candidates = list(Path("/content/src").rglob("setup.py"))
assert candidates, "zip must contain setup.py"
repo = candidates[0].parent
print("repo", repo)

## Option B — clone GitHub (skip if you used the zip)

Clones `main` on a fresh runtime, or pulls the latest commits if the repo is already here. Rerun this cell after every push.

In [ ]:
from pathlib import Path

repo = Path("/content/w4a16-linear-kernel")
if (repo / "setup.py").is_file():
    !git -C {repo} pull --ff-only
else:
    !git clone https://github.com/chris320211/w4a16-linear-kernel.git {repo}
!git -C {repo} log --oneline -1
print("repo", repo)

In [ ]:
import os
from pathlib import Path

if "repo" not in globals():
    raise RuntimeError("Run the zip cell or uncomment the clone cell so `repo` is set")

os.chdir(repo)
print("cwd", Path.cwd())
%pip install -q pytest
# Clean rebuild: a stale build/ can skip recompiling the .cu after a pull.
!rm -rf build w4a16/_C*.so
!python setup.py build_ext --inplace 2>&1 | grep -E "nvcc|error|warning|copying"
!python -m pytest tests/ -q

Operator bench (sequential baselines, CUDA events). Kernel roofline is the next cell. TinyLlama generate after that downloads the model (~2 GB).

In [ ]:
!python scripts/benchmark.py --iters 20 --warmup 5 --m 1 2 4 8

## Profile the kernel (no `ncu` needed)

Colab blocks GPU performance counters, so `ncu` fails here. These two scripts cover most of what it would tell you:

- `profile_kernel.py`: GPU time per call (`kernel_us`) from 100 back-to-back calls in a CUDA graph, with weight copies rotated so each call reads DRAM like real decode. `eager_us` is the old one-call timing (includes Python launch overhead). `%copy` compares GB/s against a measured device copy, the practical bandwidth ceiling.
- `inspect_kernel.py`: registers, spills, occupancy and the SASS instruction mix of each kernel (global load widths, barriers, shuffles, convert/FMA counts). Full SASS lands in `profile_results/sass/`.

`W4A16_DECODE` picks the `M = 1` decode kernel: `v1` original, `v2` wide loads, `v3` (default) wide loads + fast unpack. The cell runs all three back to back so they share one GPU and clock state; T4 timings drift between sessions, so compare within one run.

Add `--hot-l2` to see how much the L2 cache flatters small weights, or `--profiler torch` for Chrome traces under `profile_results/`.

In [ ]:
!python scripts/inspect_kernel.py 2>&1 | head -55
!W4A16_DECODE=v1 python scripts/profile_kernel.py --baselines fused --output profile_results/v1.txt | tail -7
!W4A16_DECODE=v2 python scripts/profile_kernel.py --baselines fused --output profile_results/v2.txt | tail -7
!python scripts/profile_kernel.py --baselines fused resident_fp16 --output profile_results/v3.txt | tail -11

## TinyLlama end-to-end

Quantize once from `TinyLlama/TinyLlama-1.1B-Chat-v1.0`. `--mode fp16` loads the original Hugging Face FP16 weights. `--mode fused` and `--mode dequant` load only the packed checkpoint onto an empty model, so peak MiB is INT4-resident VRAM. FP16 vs fused text is **quality** (quantization error), not kernel correctness.

Decode (`M = 1`) uses the fused kernel. Prefill (`M > 1`) dequantizes a temporary FP16 weight and uses cuBLAS. `dequant` mode uses cuBLAS on every call. The first FP16 or quantize run is slow because of the model download.

In [ ]:
%pip install -q transformers sentencepiece accelerate bitsandbytes

In [ ]:
!python scripts/quantize_tinyllama.py --device cuda --output checkpoints/tinyllama-w4a16.pt

In [ ]:
!python scripts/generate.py --mode fp16 --max-new-tokens 32

In [ ]:
!python scripts/generate.py --mode fused --checkpoint checkpoints/tinyllama-w4a16.pt --max-new-tokens 32

In [ ]:
!python scripts/generate.py --mode dequant --checkpoint checkpoints/tinyllama-w4a16.pt --max-new-tokens 32

Or run all three generate baselines sequentially (FP16, dequant, fused) after the checkpoint exists:

In [ ]:
!python scripts/generate.py --mode all --checkpoint checkpoints/tinyllama-w4a16.pt --max-new-tokens 32